# Matrix factorization with PyTorch

In this notebook we will write a matrix factorization model in pytorch to solve a recommendation problem. 

The MovieLens dataset (ml-latest-small) describes 5-star rating and free-text tagging activity from MovieLens, a movie recommendation service. It contains 100004 ratings and 1296 tag applications across 9125 movies. https://grouplens.org/datasets/movielens/. To get the data:

`wget http://files.grouplens.org/datasets/movielens/ml-latest-small.zip`

## MovieLens dataset

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
PATH = Path("ml-latest-small")
list(PATH.iterdir())

[PosixPath('ml-latest-small/links.csv'),
 PosixPath('ml-latest-small/tags.csv'),
 PosixPath('ml-latest-small/ratings.csv'),
 PosixPath('ml-latest-small/README.txt'),
 PosixPath('ml-latest-small/movies.csv')]

In [3]:
! head ml-latest-small/ratings.csv

userId,movieId,rating,timestamp
1,1,4.0,964982703
1,3,4.0,964981247
1,6,4.0,964982224
1,47,5.0,964983815
1,50,5.0,964982931
1,70,3.0,964982400
1,101,5.0,964980868
1,110,4.0,964982176
1,151,5.0,964984041


In [4]:
# reading a csv into pandas
data = pd.read_csv(PATH/"ratings.csv")

In [5]:
data.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


### Encoding data
We encode the data to have continuous ids for users and movies. You can think about this as a categorical encoding of our two categorical variables userId and movieId.

Also we're going to split the dataset into a training and validation set by taking the last 20% of time points as the validation set.

In [6]:
time_80 = np.quantile(data.timestamp.values, 0.8)
time_80

np.float64(1458635171.0)

In [7]:
train = data[data["timestamp"] < time_80].copy()
val = data[data["timestamp"] >= time_80].copy()

In [8]:
val.head()

,userId,movieId,rating,timestamp
1434,15,1,2.5,1510577970
1436,15,47,3.5,1510571970
1440,15,260,5.0,1510571946
1441,15,293,3.0,1510571962
1442,15,296,4.0,1510571877


In [9]:
# encoding movies and user ids with continous ids

train_user_ids = np.sort(np.unique(train.userId.values))
train_user_ids[:15]

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15])

In [10]:
# number of unique ids
num_users = len(train_user_ids)
num_users

522

In [11]:
userid2idx = {o:i for i,o in enumerate(train_user_ids)}
#userid2idx

In [12]:
train["userId"] = train["userId"].apply(lambda x: userid2idx[x])
train.head()

,userId,movieId,rating,timestamp
0,0,1,4.0,964982703
1,0,3,4.0,964981247
2,0,6,4.0,964982224
3,0,47,5.0,964983815
4,0,50,5.0,964982931


In [13]:
val["userId"] = val["userId"].apply(lambda x: userid2idx.get(x, -1)) # -1 for users not in training
val.head()

,userId,movieId,rating,timestamp
1434,14,1,2.5,1510577970
1436,14,47,3.5,1510571970
1440,14,260,5.0,1510571946
1441,14,293,3.0,1510571962
1442,14,296,4.0,1510571877


In [14]:
val = val[val["userId"] >= 0].copy()
val.head()

,userId,movieId,rating,timestamp
1434,14,1,2.5,1510577970
1436,14,47,3.5,1510571970
1440,14,260,5.0,1510571946
1441,14,293,3.0,1510571962
1442,14,296,4.0,1510571877


In [15]:
# now encoding movieId
train_movie_ids = np.sort(np.unique(train.movieId.values))
num_items = len(train_movie_ids)
print(num_items)
train_movie_ids[:15]

7867


array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15])

In [16]:
movieid2idx = {o:i for i,o in enumerate(train_movie_ids)}
train["movieId"] = train["movieId"].apply(lambda x: movieid2idx[x])
val["movieId"] = val["movieId"].apply(lambda x: movieid2idx.get(x, -1))

In [17]:
val = val[val["movieId"] >= 0].copy()
val.head()

,userId,movieId,rating,timestamp
1434,14,0,2.5,1510577970
1436,14,43,3.5,1510571970
1440,14,224,5.0,1510571946
1441,14,254,3.0,1510571962
1442,14,257,4.0,1510571877


In [18]:
val.shape

(1311, 4)

## Embedding layer

An embedding layer enables us to encode users and items into vectors. Every user and item is going to have a (unique) vector. These vectors are parameters of the model that are going to be learned in the optimization process. Ideally, the embeddings capture properties of the data by placing similar users (items) in close together in the embedding space.

In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [20]:
# an Embedding module containing 10 users or items embedding size 3
# embedding will be initialized at random
embed = nn.Embedding(10, 3)
embed.weight

Parameter containing:
tensor([[ 0.2578,  0.5547,  0.7995],
        [ 0.8325, -0.5650,  0.1169],
        [ 1.2176,  0.3814,  0.8801],
        [ 0.5089,  0.0590, -0.7743],
        [ 0.8450,  1.1903,  1.8592],
        [-0.0536, -1.2939, -1.5800],
        [ 0.7453,  2.4825,  0.5235],
        [-0.8962, -1.1931, -0.9469],
        [ 1.0475, -0.1591,  1.8938],
        [-1.2855, -0.0795, -1.2413]], requires_grad=True)

In [21]:
# given a list of ids we can "look up" the embedding corresponing to each id
# can you see that some vectors are the same?
a = torch.LongTensor([[1,0,1,4,5,1]])
embed(a)

tensor([[[ 0.8325, -0.5650,  0.1169],
         [ 0.2578,  0.5547,  0.7995],
         [ 0.8325, -0.5650,  0.1169],
         [ 0.8450,  1.1903,  1.8592],
         [-0.0536, -1.2939, -1.5800],
         [ 0.8325, -0.5650,  0.1169]]], grad_fn=<EmbeddingBackward0>)

## Matrix factorization model

In [22]:
class MF(nn.Module):
    def __init__(self, num_users, num_items, emb_size=100):
        super(MF, self).__init__()
        self.user_emb = nn.Embedding(num_users, emb_size) #Creates a user embedding matrix of size (num_users, emb_size)
        self.item_emb = nn.Embedding(num_items, emb_size) #Creates an item embedding matrix of size (num_items, emb_size)
        # initlializing weights
        self.user_emb.weight.data.uniform_(0,0.05)
        self.item_emb.weight.data.uniform_(0,0.05)
        
    def forward(self, u, v):
        ## this is where you define your model! 
        u = self.user_emb(u)
        v = self.item_emb(v)
        return (u*v).sum(1)  

## Debugging MF model

In [23]:
df = pd.DataFrame({"userId": [0, 0, 1, 1, 3, 4], 
                   "movieId": [0, 1, 2, 1, 3, 0], 
                   "rating": [4, 5, 3, 1, 3, 4]})
df

,userId,movieId,rating
0,0,0,4
1,0,1,5
2,1,2,3
3,1,1,1
4,3,3,3
5,4,0,4


In [24]:
users = torch.LongTensor(df.userId.values)
users

/var/folders/09/4h25jfnn06q73576vknj8hz40000gn/T/ipykernel_48761/1539904836.py:1: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/ec2-user/croot/libtorch_1769007892424/work/torch/csrc/utils/tensor_numpy.cpp:209.)
  users = torch.LongTensor(df.userId.values)


tensor([0, 0, 1, 1, 3, 4])

In [25]:
items = torch.LongTensor(df.movieId.values)
items

tensor([0, 1, 2, 1, 3, 0])

**Why Use LongTensor?** 

**Embedding Layers Require LongTensor as Input**

When using nn.Embedding(), the input must be of type torch.LongTensor (or torch.int64), because embeddings use integer indices.

Example code:



In [26]:
#user_emb = nn.Embedding(num_users, emb_size)
#user_vector = user_emb(users)  # users must be LongTensor

If you pass a FloatTensor, it would cause an error.

User IDs are discrete categorical values, so they should be stored as integers rather than floats.

In [27]:
num_users = 5
num_items = 4
emb_size = 3

user_emb = nn.Embedding(num_users, emb_size)
item_emb = nn.Embedding(num_items, emb_size)
users = torch.LongTensor(df.userId.values)
items = torch.LongTensor(df.movieId.values)

In [28]:
U = user_emb(users)
V = item_emb(items)

In [29]:
U

tensor([[ 0.7849,  2.0522,  1.1976],
        [ 0.7849,  2.0522,  1.1976],
        [-0.3844, -0.5468,  1.8748],
        [-0.3844, -0.5468,  1.8748],
        [-0.2235,  0.2817,  0.6235],
        [ 0.2786,  0.3104, -0.3779]], grad_fn=<EmbeddingBackward0>)

In [30]:
V

tensor([[ 1.0307,  0.3820, -0.9049],
        [-0.0097,  0.5865, -0.6905],
        [ 1.3714,  0.4210, -1.1930],
        [-0.0097,  0.5865, -0.6905],
        [ 2.0409, -1.1359, -1.5578],
        [ 1.0307,  0.3820, -0.9049]], grad_fn=<EmbeddingBackward0>)

In [31]:
# element wise multiplication
U*V 

tensor([[ 0.8089,  0.7840, -1.0836],
        [-0.0076,  1.2037, -0.8269],
        [-0.5272, -0.2302, -2.2366],
        [ 0.0037, -0.3207, -1.2946],
        [-0.4561, -0.3199, -0.9713],
        [ 0.2871,  0.1186,  0.3419]], grad_fn=<MulBackward0>)

In [32]:
# what we want is a dot product per row
(U*V).sum(1) 

tensor([ 0.5093,  0.3692, -2.9940, -1.6116, -1.7474,  0.7477],
       grad_fn=<SumBackward1>)

## Training MF model

In [33]:
num_users = len(train.userId.unique())
num_items = len(train.movieId.unique())
print(num_users, num_items) 

522 7867


In [34]:
# here we are not using data loaders because our data fits well in memory
def train_epochs(model, epochs=10, lr=0.01, wd=0.0):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr,
                                 weight_decay=wd)
    for i in range(epochs):
        model.train()
        users = torch.LongTensor(train.userId.values)  #.cuda()
        items = torch.LongTensor(train.movieId.values) #.cuda()
        ratings = torch.FloatTensor(train.rating.values)  #.cuda()
    
        y_hat = model(users, items)
        loss = F.mse_loss(y_hat, ratings)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        testloss = valid_loss(model)
        print("train loss %.3f valid loss %.3f" % (loss.item(), testloss)) 

In [35]:
def valid_loss(model):
    model.eval()
    users = torch.LongTensor(val.userId.values) # .cuda()
    items = torch.LongTensor(val.movieId.values) #.cuda()
    ratings = torch.FloatTensor(val.rating.values) #.cuda()
    y_hat = model(users, items)
    loss = F.mse_loss(y_hat, ratings)
    return loss.item()

In [36]:
num_users = len(train_user_ids)
num_users

522

In [37]:
num_items = len(train_movie_ids)
num_items

7867

In [38]:
model = MF(num_users, num_items, emb_size=100)  # if you have a GPU .cuda()

In [39]:
train_epochs(model, epochs=20, lr=0.1, wd=1e-5)

train loss 12.942 valid loss 5.060
train loss 4.867 valid loss 2.695
train loss 2.532 valid loss 4.524
train loss 2.988 valid loss 1.454
train loss 0.852 valid loss 1.822
train loss 1.874 valid loss 2.665
train loss 2.684 valid loss 2.353
train loss 2.127 valid loss 1.410
train loss 1.069 valid loss 1.135
train loss 0.974 valid loss 1.755
train loss 1.624 valid loss 1.753
train loss 1.307 valid loss 1.179
train loss 0.774 valid loss 1.005
train loss 0.966 valid loss 1.182
train loss 1.344 valid loss 1.243
train loss 1.307 valid loss 1.091
train loss 0.916 valid loss 0.995
train loss 0.675 valid loss 1.147
train loss 0.871 valid loss 1.261
train loss 1.022 valid loss 1.098


In [40]:
train_epochs(model, epochs=15, lr=0.01, wd=1e-5)

train loss 0.799 valid loss 0.930
train loss 0.631 valid loss 0.905
train loss 0.643 valid loss 0.907
train loss 0.664 valid loss 0.908
train loss 0.649 valid loss 0.912
train loss 0.623 valid loss 0.916
train loss 0.607 valid loss 0.913
train loss 0.602 valid loss 0.899
train loss 0.600 valid loss 0.878
train loss 0.592 valid loss 0.857
train loss 0.580 valid loss 0.844
train loss 0.569 valid loss 0.838
train loss 0.563 valid loss 0.839
train loss 0.561 valid loss 0.842
train loss 0.558 valid loss 0.847


In [41]:
train_epochs(model, epochs=15, lr=0.001, wd=1e-5)

train loss 0.550 valid loss 0.845
train loss 0.543 valid loss 0.845
train loss 0.537 valid loss 0.845
train loss 0.533 valid loss 0.845
train loss 0.529 valid loss 0.846
train loss 0.526 valid loss 0.847
train loss 0.524 valid loss 0.847
train loss 0.522 valid loss 0.848
train loss 0.520 valid loss 0.848
train loss 0.518 valid loss 0.848
train loss 0.516 valid loss 0.847
train loss 0.514 valid loss 0.847
train loss 0.511 valid loss 0.847
train loss 0.509 valid loss 0.846
train loss 0.507 valid loss 0.846


## MF with bias

In [42]:
class MF_bias(nn.Module):
    def __init__(self, num_users, num_items, emb_size=100):
        super(MF_bias, self).__init__()
        self.user_emb = nn.Embedding(num_users, emb_size)
        self.user_bias = nn.Embedding(num_users, 1)
        self.item_emb = nn.Embedding(num_items, emb_size)
        self.item_bias = nn.Embedding(num_items, 1)
        # init 
        self.user_emb.weight.data.uniform_(0,0.05)
        self.item_emb.weight.data.uniform_(0,0.05)
        self.user_bias.weight.data.uniform_(-0.01,0.01)
        self.item_bias.weight.data.uniform_(-0.01,0.01)
        
    def forward(self, u, v):
        U = self.user_emb(u)
        V = self.item_emb(v)
        b_u = self.user_bias(u).squeeze()
        b_v = self.item_bias(v).squeeze()
        return (U*V).sum(1) +  b_u  + b_v

In [43]:
model = MF_bias(num_users, num_items, emb_size=100) #.cuda()

In [44]:
train_epochs(model, epochs=15, lr=0.1, wd=1e-5)

train loss 12.942 valid loss 4.379
train loss 4.132 valid loss 3.818
train loss 3.738 valid loss 3.650
train loss 2.270 valid loss 1.244
train loss 0.770 valid loss 1.777
train loss 1.846 valid loss 2.469
train loss 2.503 valid loss 2.260
train loss 2.108 valid loss 1.546
train loss 1.281 valid loss 1.153
train loss 0.948 valid loss 1.417
train loss 1.273 valid loss 1.571
train loss 1.277 valid loss 1.278
train loss 0.898 valid loss 1.045
train loss 0.819 valid loss 1.069
train loss 1.043 valid loss 1.143


In [45]:
train_epochs(model, epochs=10, lr=0.01, wd=1e-5)

train loss 1.190 valid loss 0.975
train loss 0.883 valid loss 0.933
train loss 0.713 valid loss 0.956
train loss 0.664 valid loss 0.989
train loss 0.681 valid loss 0.997
train loss 0.704 valid loss 0.977
train loss 0.702 valid loss 0.945
train loss 0.682 valid loss 0.914
train loss 0.659 valid loss 0.894
train loss 0.643 valid loss 0.885


In [46]:
train_epochs(model, epochs=10, lr=0.001, wd=1e-5)

train loss 0.638 valid loss 0.881
train loss 0.630 valid loss 0.878
train loss 0.624 valid loss 0.875
train loss 0.618 valid loss 0.873
train loss 0.613 valid loss 0.871
train loss 0.609 valid loss 0.869
train loss 0.605 valid loss 0.868
train loss 0.601 valid loss 0.866
train loss 0.598 valid loss 0.865
train loss 0.595 valid loss 0.864


In [47]:
train_epochs(model, epochs=10, lr=0.001, wd=1e-5)

train loss 0.593 valid loss 0.861
train loss 0.590 valid loss 0.859
train loss 0.587 valid loss 0.858
train loss 0.585 valid loss 0.857
train loss 0.583 valid loss 0.857
train loss 0.581 valid loss 0.857
train loss 0.579 valid loss 0.857
train loss 0.576 valid loss 0.857
train loss 0.574 valid loss 0.858
train loss 0.572 valid loss 0.858


In [48]:
train_epochs(model, epochs=10, lr=0.001, wd=1e-5)

train loss 0.570 valid loss 0.859
train loss 0.568 valid loss 0.860
train loss 0.566 valid loss 0.860
train loss 0.564 valid loss 0.861
train loss 0.561 valid loss 0.862
train loss 0.559 valid loss 0.862
train loss 0.557 valid loss 0.863
train loss 0.555 valid loss 0.863
train loss 0.552 valid loss 0.863
train loss 0.550 valid loss 0.863


Note that these models are sensitive to weight initialization, optimization algorithm and regularization.

## Lab
* Can we change the first model to predict numbers in a particular range? Hint: sigmoid would create numbers between 0 and 1. Would this improve the model?
* Would a different Loss function improve results? What about absolute value instead of F.mse_loss?

In [ ]:
model = MF(num_users, num_items, emb_size=100)  # if you have a GPU .cuda()

class MF_range(nn.Module):
    def __init__(self, num_users, num_items, min_user, max_user, min_item, max_item, emb_size=100):
        super(MF_bias, self).__init__()
        self.user_emb = nn.Embedding(num_users, emb_size)
        self.item_emb = nn.Embedding(num_items, emb_size)

        self.user_emb.weight.data.uniform_(min_user,max_user)
        self.item_emb.weight.data.uniform_(min_item,max_item)
        
    def forward(self, u, v):
        U = self.user_emb(u)
        V = self.item_emb(v)
        return (U*V).sum(1)

In [50]:
train_epochs(model, epochs=10, lr=0.001, wd=1e-5)

train loss 12.944 valid loss 13.106
train loss 12.909 valid loss 13.070
train loss 12.872 valid loss 13.032
train loss 12.835 valid loss 12.993
train loss 12.795 valid loss 12.952
train loss 12.755 valid loss 12.910
train loss 12.713 valid loss 12.866
train loss 12.669 valid loss 12.821
train loss 12.625 valid loss 12.775
train loss 12.578 valid loss 12.727


# References
* This notebook is based on Lesson 5 of Jeremy Howard's Deep Learning Course